<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week6_day5_Exercises_XP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-projet : Analyse de Sentiment avec BERT Fine-Tuning
Ce notebook présente l'affinage d'un modèle BERT pour la classification de sentiments sur le jeu de données IMDB.

In [ ]:
!pip install --upgrade --quiet tensorflow-datasets transformers accelerate evaluate tensorflow protobuf

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

# Paramètres globaux
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 2

# Chargement du tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
print('Tokenizer chargé.')

In [ ]:
# Chargement et préparation des données
(ds_train, ds_test), ds_info = tfds.load(
    'imdb_reviews',
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)

def tf_encode(text, label):
    def _encode(t):
        t_str = t.numpy().decode('utf-8')
        enc = tokenizer(t_str, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='tf')
        return enc['input_ids'][0], enc['attention_mask'][0], enc['token_type_ids'][0]

    input_ids, attention_mask, token_type_ids = tf.py_function(
        func=_encode, inp=[text], Tout=[tf.int32, tf.int32, tf.int32]
    )
    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'token_type_ids': token_type_ids}, label

train_ds = ds_train.map(tf_encode).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = ds_test.map(tf_encode).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Initialisation du modèle
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

# Entraînement (limité à 1 époque pour la démo si besoin, ou EPOCHS)
model.fit(train_ds.take(500), validation_data=test_ds.take(100), epochs=EPOCHS)

### Visualisation des performances
Il est important de visualiser les courbes de perte (loss) et de précision (accuracy) pour vérifier si le modèle converge correctement ou s'il commence à surapprendre (overfitting).

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Précision Entraînement')
    plt.plot(epochs_range, val_acc, label='Précision Validation')
    plt.title('Précision')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Perte Entraînement')
    plt.plot(epochs_range, val_loss, label='Perte Validation')
    plt.title('Perte')
    plt.legend()
    plt.show()

if 'history' in locals():
    plot_history(history)

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors='tf', max_length=MAX_LENGTH, padding='max_length', truncation=True)
    logits = model(inputs).logits
    probs = tf.nn.softmax(logits, axis=-1)
    label = 'Positif' if tf.argmax(probs, axis=-1).numpy()[0] == 1 else 'Négatif'
    return label

print(predict_sentiment('Ce film était absolument fantastique !'))

### Évaluation finale
Calculons les métriques finales sur l'ensemble de test pour confirmer la robustesse de notre assistant de sentiment.

In [ ]:
results = model.evaluate(test_ds.take(200))
print(f"Perte sur le test : {results[0]:.4f}")
print(f"Précision sur le test : {results[1]:.4f}")

### Sauvegarde du modèle
Une fois l'entraînement terminé, nous sauvegardons les poids du modèle et le tokenizer pour pouvoir les réutiliser plus tard sans avoir à tout ré-entraîner.

In [ ]:
# Sauvegarde locale
model.save_pretrained('./mon_modele_bert_sentiment')
tokenizer.save_pretrained('./mon_modele_bert_sentiment')
print("Modèle et tokenizer sauvegardés dans le dossier './mon_modele_bert_sentiment'")

### Démonstration Finale : Cas d'usage Support Client
Appliquons notre modèle à un exemple concret de retour client pour voir comment il se comporte dans un scénario de prévention du churn.

In [ ]:
# Exemple de texte client
feedback_client = "Le service était correct mais l'attente au téléphone est inacceptable, je vais probablement résilier mon abonnement."

# Prédiction
resultat = predict_sentiment(feedback_client)
print(f"Texte : {feedback_client}")
print(f"Sentiment détecté : {resultat}")

if resultat == 'Négatif':
    print("Action suggérée : Alerte prioritaire pour l'équipe Customer Success.")

### Conclusion
Le projet est maintenant complet. Nous avons :
1. Chargé et préparé les données IMDB.
2. Configuré et entraîné un modèle BERT.
3. Visualisé les courbes d'apprentissage.
4. Évalué la précision sur un jeu de test.
5. Sauvegardé le modèle pour une utilisation future.
6. Testé le modèle sur un cas d'usage réel de support client.

### Questions de Réflexion

1. **Quel levier a le plus amélioré les résultats ?**
*Réponse :* Généralement, c'est l'utilisation d'un modèle pré-entraîné (Transfer Learning) qui permet d'atteindre une haute précision rapidement.

2. **Où ajouteriez-vous des garde-fous avant le déploiement ?**
*Réponse :* Il faudrait ajouter un seuil de confiance (ex: n'accepter que si probabilité > 80%) et un filtrage des entrées trop courtes ou non pertinentes.

3. **Quelles sont les parties prenantes qui bénéficieraient le plus ?**
*Réponse :* Le responsable du support pour automatiser le tri des plaintes et le chef de produit pour analyser les retours clients à grande échelle.